# Transfer learning: feature extraction vs fine-tuning
Same 2,000 training photos, three ways (3 seeds each): a small CNN trained from scratch (with augmentation),
VGG16's frozen convolutional base with a new dense head (feature extraction), and the same model with the last
convolution block unfrozen for the second half of training (fine-tuning). All are scored on 1,000 test photos.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's start-up messages
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"  # take GPU memory only as needed
os.environ["TF_CUDNN_WORKSPACE_LIMIT_IN_MB"] = "256"   # the GPU is shared; keep scratch memory small
import warnings
warnings.filterwarnings("ignore", category=UserWarning)   # Keras notices about tf.data
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers

tf.get_logger().setLevel("ERROR")
HERE = Path.cwd().parent if Path.cwd().name == ".logs" else Path.cwd()   # the Note folder, also from .logs/
(HERE / "data").mkdir(exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

In [ ]:
# The Kaggle / Microsoft cats-vs-dogs photos: 12,500 cats and 12,500 dogs, one folder per class.
DATA = Path.home() / "datasets" / "PetImages"
URL = ("https://download.microsoft.com/download/3/E/1/"
       "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")
if not DATA.exists():
    zip_path = keras.utils.get_file("kagglecatsanddogs_5340.zip", URL, cache_dir=Path.home(), cache_subdir="datasets")
    import shutil
    shutil.unpack_archive(zip_path, DATA.parent)
# Some files are damaged or are not JPEGs at all; delete them, as the official Keras example does.
removed = 0
for path in DATA.glob("*/*"):
    with open(path, "rb") as f:
        is_jpeg = b"JFIF" in f.peek(10)
    if not is_jpeg:
        path.unlink()
        removed += 1
counts = {d.name: len(list(d.iterdir())) for d in sorted(DATA.iterdir())}
print("removed now:", removed, "| images per class:", counts, "| total:", sum(counts.values()))

In [ ]:
# A small subset, as when data is scarce: 1,000 cats and 1,000 dogs for training,
# 500 + 500 for validation and 500 + 500 for testing, picked at random once (seed 0).
# Symbolic links put them in the folder layout that image_dataset_from_directory expects.
SMALL = Path.home() / "datasets" / "cats_vs_dogs_small"
if not SMALL.exists():
    rng = np.random.default_rng(0)
    for cls in ("Cat", "Dog"):
        files = sorted(DATA.glob(f"{cls}/*.jpg"), key=lambda p: int(p.stem))
        pick = rng.choice(len(files), 2000, replace=False)
        for split, idx in (("train", pick[:1000]), ("validation", pick[1000:1500]), ("test", pick[1500:])):
            folder = SMALL / split / cls.lower()
            folder.mkdir(parents=True, exist_ok=True)
            for i in idx:
                (folder / files[i].name).symlink_to(files[i])
print({split: sum(1 for _ in (SMALL / split).glob("*/*")) for split in ("train", "validation", "test")})

In [ ]:
def load(split, shuffle, batch_size=32):
    ds = keras.utils.image_dataset_from_directory(SMALL / split, label_mode="binary", image_size=(150, 150),
                                                  batch_size=batch_size, shuffle=shuffle, seed=1)
    return ds.cache().prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = load("train", True), load("validation", False), load("test", False)

## 1. The convolutional base of VGG16

In [ ]:
from keras.applications.vgg16 import VGG16, preprocess_input
full = VGG16(weights="imagenet")                          # with its dense top, for the parameter count
conv_base = VGG16(weights="imagenet", include_top=False, input_shape=(150, 150, 3))
print(f"full VGG16: {full.count_params():,} parameters; convolutional base only: {conv_base.count_params():,}")
print("base output for a 150 x 150 photo:", conv_base.output_shape)
print([l.name for l in conv_base.layers][-5:])
del full

In [ ]:
def make_transfer_model():
    base = VGG16(weights="imagenet", include_top=False, input_shape=(150, 150, 3))
    base.trainable = False                                 # freeze: its weights will not change
    model = keras.Sequential([
        keras.Input(shape=(150, 150, 3)),
        layers.Lambda(preprocess_input),                   # VGG16's own pixel preparation
        base,
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model, base

m, b = make_transfer_model()
m.summary()
trainable = sum(int(np.prod(w.shape)) for w in m.trainable_weights)
print(f"total {m.count_params():,}; trainable after freezing the base {trainable:,}")
b.trainable = True
print(f"trainable if the base were not frozen {sum(int(np.prod(w.shape)) for w in m.trainable_weights):,}")
for layer in b.layers:
    layer.trainable = layer.name.startswith("block5")
print(f"trainable with only block 5 unfrozen {sum(int(np.prod(w.shape)) for w in m.trainable_weights):,}")
del m, b

## 2. A shortcut that changes nothing
Blocks 1 to 4 of VGG16 stay frozen in both transfer methods, and the photos are not augmented, so those blocks turn
each photo into exactly the same numbers in every epoch. We run every photo through them once, store the result,
and train only block 5 and the new head on it. The model is the same; only the repeated work is skipped.

In [ ]:
lower = keras.Model(conv_base.input, conv_base.get_layer("block4_pool").output)   # blocks 1-4
def lower_features(ds):
    xs, ys = [], []
    for x, y in ds:
        xs.append(lower.predict(preprocess_input(x.numpy()), verbose=0))
        ys.append(y.numpy())
    return np.concatenate(xs), np.concatenate(ys)
F = {split: lower_features(ds) for split, ds in (("train", train_ds), ("val", val_ds), ("test", test_ds))}
print("stored output of block 4 for each photo:", F["train"][0].shape[1:])

def make_upper(seed):
    # block 5 of VGG16 (three 3 x 3 convolutions with 512 filters, then pooling), with its ImageNet weights,
    # then the new head: Flatten, Dense(256), Dense(1)
    keras.utils.set_random_seed(seed)
    inp = keras.Input(shape=F["train"][0].shape[1:])
    block5 = [layers.Conv2D(512, 3, padding="same", activation="relu", name=n)
              for n in ("block5_conv1", "block5_conv2", "block5_conv3")]
    y = inp
    for layer in block5:
        y = layer(y)
    y = layers.MaxPooling2D(2, name="block5_pool")(y)
    y = layers.Flatten()(y)
    y = layers.Dense(256, activation="relu")(y)
    out = layers.Dense(1, activation="sigmoid")(y)
    model = keras.Model(inp, out)
    for layer in block5:
        layer.set_weights(conv_base.get_layer(layer.name).get_weights())
        layer.trainable = False                                   # frozen to start with
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model, block5

# Check: blocks 1-4 stored + block 5 rebuilt = the original base, on real photos
check, _ = make_upper(0)
x, _ = next(iter(test_ds))
upper5 = keras.Model(check.input, check.get_layer("block5_pool").output)
same = np.allclose(upper5.predict(lower.predict(preprocess_input(x.numpy()), verbose=0), verbose=0),
                   conv_base.predict(preprocess_input(x.numpy()), verbose=0), atol=1e-3)
print("split model gives the same output as VGG16's base:", same)

## 3. Train the three ways

In [ ]:
def make_scratch_model():                                   # the small CNN of the augmentation Note
    model = keras.Sequential([keras.Input(shape=(150, 150, 3)),
        layers.RandomFlip("horizontal"), layers.RandomRotation(0.1), layers.RandomZoom(0.2),
        layers.Rescaling(1 / 255)])
    for filters in (32, 32, 64):
        model.add(layers.Conv2D(filters, 3, activation="relu"))
        model.add(layers.MaxPooling2D(2))
    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation="relu"))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
    return model

import time
KEYS = ("accuracy", "val_accuracy", "loss", "val_loss")
rows, scores = [], []
def record(name, seed, h, offset=0):
    for e in range(len(h.history["loss"])):
        rows.append(dict(method=name, seed=seed, epoch=offset + e + 1, **{k: h.history[k][e] for k in KEYS}))
fit_kw = dict(batch_size=32, validation_data=F["val"], verbose=0)

for seed in (0, 1, 2):
    t = time.time()
    # (a) from scratch, with augmentation, 60 epochs
    keras.utils.set_random_seed(seed)
    scratch = make_scratch_model()
    record("scratch", seed, scratch.fit(train_ds, epochs=60, validation_data=val_ds, verbose=0))
    scores.append(dict(method="scratch", seed=seed, test_accuracy=scratch.evaluate(test_ds, verbose=0)[1]))
    # (b) feature extraction: the whole base frozen, train the new head for 20 epochs
    fe, _ = make_upper(seed)
    record("feature_extraction", seed, fe.fit(*F["train"], epochs=20, **fit_kw))
    scores.append(dict(method="feature_extraction", seed=seed, test_accuracy=fe.evaluate(*F["test"], verbose=0)[1]))
    # (c) fine-tuning: the same first 10 epochs, then unfreeze block 5 and train 10 more with a small learning rate
    ft, block5 = make_upper(seed)
    record("fine_tuning", seed, ft.fit(*F["train"], epochs=10, **fit_kw))
    for layer in block5:
        layer.trainable = True
    ft.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-5), loss="binary_crossentropy", metrics=["accuracy"])
    record("fine_tuning", seed, ft.fit(*F["train"], epochs=10, **fit_kw), offset=10)
    scores.append(dict(method="fine_tuning", seed=seed, test_accuracy=ft.evaluate(*F["test"], verbose=0)[1]))
    print(f"seed {seed}:", {s["method"]: round(s["test_accuracy"], 3) for s in scores[-3:]}, f"({time.time() - t:.0f} s)")

hist = pd.DataFrame(rows).round(4)
hist.to_csv(HERE / "data" / "history.csv", index=False)
scores = pd.DataFrame(scores).round(4)
scores.to_csv(HERE / "data" / "test_accuracy.csv", index=False)

In [ ]:
print(scores.groupby("method").test_accuracy.agg(["mean", "min", "max"]).round(3))
mean = hist.groupby(["method", "epoch"])[list(KEYS)].mean()
for name in ("scratch", "feature_extraction", "fine_tuning"):
    g = mean.loc[name]
    last = g.index.max()
    print(f"{name}: epoch {last} train {g.accuracy.iloc[-1]:.3f} val {g.val_accuracy.iloc[-1]:.3f};"
          f" val at epoch 10 {g.val_accuracy.loc[10]:.3f}; best val {g.val_accuracy.max():.3f} (epoch {g.val_accuracy.idxmax()})")